# 🏆 Football Prediction App — 01: Data Exploration

**Proje:** 2026 FIFA Dünya Kupası Tahmin Motoru  
**Notebook amacı:** Tüm CSV dosyalarını yüklemek, şema doğrulaması yapmak, takım isimlerini standardize etmek ve veriye genel bakış sunmak.

---
### Notebook bölümleri
1. Kütüphane importları
2. Dosya yolları & config
3. CSV yükleme fonksiyonları
4. Şema doğrulama
5. Takım ismi standardizasyonu
6. Her dosyaya genel bakış
7. Kalite özeti

---
## 1. Kütüphane İmportları

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 120)

print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)

---
## 2. Dosya Yolları & Config

In [ ]:
# Bu notebook notebooks/ klasöründen çalışır.
# Relative path ile data/raw'a erişiyoruz.
RAW_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw')
# Eğer notebook doğrudan football_prediction_app/ içinden çalışıyorsa:
if not os.path.isdir(RAW_DIR):
    RAW_DIR = os.path.join(os.getcwd(), '..', 'data', 'raw')
if not os.path.isdir(RAW_DIR):
    RAW_DIR = os.path.join(os.getcwd(), 'data', 'raw')

FILES = {
    'results'       : os.path.join(RAW_DIR, 'results.csv'),
    'elo'           : os.path.join(RAW_DIR, 'elo_ratings_wc2026.csv'),
    'group_fixtures': os.path.join(RAW_DIR, 'GROUP_FIXTURES.CSV'),
    'knockout_slots': os.path.join(RAW_DIR, 'KNOCKOUT_SLOTS.CSV'),
    'goalscorers'   : os.path.join(RAW_DIR, 'goalscorers.csv'),
    'shootouts'     : os.path.join(RAW_DIR, 'shootouts.csv'),
    'former_names'  : os.path.join(RAW_DIR, 'former_names.csv'),
    'players'       : os.path.join(RAW_DIR, 'players_data-2025_2026.csv'),
}

for name, path in FILES.items():
    status = '✅' if os.path.isfile(path) else '❌ MISSING'
    size   = f"{os.path.getsize(path)/1024:.1f} KB" if os.path.isfile(path) else ''
    print(f"{status}  {name:<18} {size}")

---
## 3. CSV Yükleme Fonksiyonları

In [ ]:
def load_csv(key: str, parse_dates: list = None, dtype: dict = None) -> pd.DataFrame:
    """Güvenli CSV yükleyici — dosya yoksa veya bozuksa boş DataFrame döner."""
    path = FILES.get(key, '')
    if not os.path.isfile(path):
        print(f"[WARN] {key} dosyası bulunamadı: {path}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, parse_dates=parse_dates, dtype=dtype, low_memory=False)
        print(f"[OK]  {key:<18} → {df.shape[0]:>7,} satır × {df.shape[1]} kolon")
        return df
    except Exception as e:
        print(f"[ERROR] {key} yüklenemedi: {e}")
        return pd.DataFrame()


print("=" * 55)
print("Dosya yükleme başlıyor...")
print("=" * 55)

results        = load_csv('results',        parse_dates=['date'])
elo            = load_csv('elo',            parse_dates=['snapshot_date'])
group_fixtures = load_csv('group_fixtures', parse_dates=['date_utc'])
knockout_slots = load_csv('knockout_slots', parse_dates=['date_utc'])
goalscorers    = load_csv('goalscorers',    parse_dates=['date'])
shootouts      = load_csv('shootouts',      parse_dates=['date'])
former_names   = load_csv('former_names',   parse_dates=['start_date', 'end_date'])
players        = load_csv('players')

print("=" * 55)

---
## 4. Şema Doğrulama

In [ ]:
EXPECTED_SCHEMAS = {
    'results'       : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral'],
    'elo'           : ['year', 'snapshot_date', 'country', 'rank', 'rating', 'confederation'],
    'group_fixtures': ['match_id', 'group', 'home_team', 'away_team', 'date_utc', 'venue'],
    'knockout_slots': ['match_id', 'round', 'date_utc', 'venue', 'slot_home', 'slot_away'],
    'goalscorers'   : ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute'],
    'shootouts'     : ['date', 'home_team', 'away_team', 'winner'],
    'former_names'  : ['current', 'former'],
}

dfs = {
    'results': results, 'elo': elo, 'group_fixtures': group_fixtures,
    'knockout_slots': knockout_slots, 'goalscorers': goalscorers,
    'shootouts': shootouts, 'former_names': former_names,
}

print("ŞEMA DOĞRULAMA RAPORU")
print("=" * 60)
all_ok = True
for name, expected_cols in EXPECTED_SCHEMAS.items():
    df = dfs.get(name, pd.DataFrame())
    if df.empty:
        print(f"  ⚠️  {name:<20} — DataFrame boş, atlandı")
        continue
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        print(f"  ❌  {name:<20} — eksik kolonlar: {missing}")
        all_ok = False
    else:
        print(f"  ✅  {name:<20} — tüm beklenen kolonlar mevcut")

print("=" * 60)
print("Sonuç:", "Tüm şemalar geçerli ✅" if all_ok else "Bazı kolonlar eksik ⚠️")

---
## 5. Takım İsmi Standardizasyonu

Farklı kaynaklarda aynı ülke farklı isimlerle geçebilir.  
Merkezi `TEAM_NAME_MAP` sözlüğü tüm DataFramelere uygulanır.

In [ ]:
# Merkezi isim eşleştirme sözlüğü
TEAM_NAME_MAP = {
    # Eski isim          : Standart isim
    "Czech Republic"     : "Czechia",
    "Ivory Coast"        : "Côte d'Ivoire",
    "Cape Verde"         : "Cabo Verde",
    "USA"                : "United States",
    "Korea Republic"     : "South Korea",
    "Republic of Ireland": "Ireland",
    "North Ireland"      : "Northern Ireland",
    "Kyrgyz Republic"    : "Kyrgyzstan",
    "Trinidad and Tobago": "Trinidad & Tobago",
    "St Kitts and Nevis" : "Saint Kitts and Nevis",
    "St Lucia"           : "Saint Lucia",
    "St Vincent / Grenadines": "Saint Vincent and the Grenadines",
}

def standardize_team_names(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Verilen kolon listesine TEAM_NAME_MAP uygular."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            before = df[col].nunique()
            df[col] = df[col].replace(TEAM_NAME_MAP)
            after = df[col].nunique()
            if before != after:
                print(f"  [{col}] unique takım: {before} → {after} (birleştirme gerçekleşti)")
    return df


print("Standardizasyon uygulanıyor...")
if not results.empty:
    results = standardize_team_names(results, ['home_team', 'away_team'])
if not elo.empty:
    elo = standardize_team_names(elo, ['country'])
if not group_fixtures.empty:
    group_fixtures = standardize_team_names(group_fixtures, ['home_team', 'away_team'])
if not goalscorers.empty:
    goalscorers = standardize_team_names(goalscorers, ['home_team', 'away_team', 'team'])
if not shootouts.empty:
    shootouts = standardize_team_names(shootouts, ['home_team', 'away_team', 'winner'])

print("Standardizasyon tamamlandı ✅")

---
## 6. Her Dosyaya Genel Bakış

### 6.1 results.csv — Tarihi Maç Sonuçları

In [ ]:
if not results.empty:
    print(f"Toplam maç sayısı    : {len(results):,}")
    print(f"Tarih aralığı        : {results['date'].min().date()}  →  {results['date'].max().date()}")
    print(f"Benzersiz home takım : {results['home_team'].nunique()}")
    print(f"Benzersiz away takım : {results['away_team'].nunique()}")
    print(f"Turnuva türleri      : {results['tournament'].nunique()}")
    print(f"Nötr saha oranı      : {results['neutral'].mean()*100:.1f}%")
    print("\nİlk 5 satır:")
    display(results.head())

In [ ]:
if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Yıllara göre maç sayısı
    results['year'] = results['date'].dt.year
    year_counts = results.groupby('year').size()
    axes[0].bar(year_counts.index, year_counts.values, color='steelblue', width=0.8)
    axes[0].set_title('Yıllara Göre Maç Sayısı')
    axes[0].set_xlabel('Yıl')
    axes[0].set_ylabel('Maç Sayısı')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    # Gol dağılımı
    total_goals = results['home_score'].fillna(0) + results['away_score'].fillna(0)
    goal_counts = total_goals.value_counts().sort_index().head(12)
    axes[1].bar(goal_counts.index.astype(int), goal_counts.values, color='coral')
    axes[1].set_title('Maç Başına Toplam Gol Dağılımı')
    axes[1].set_xlabel('Toplam Gol')
    axes[1].set_ylabel('Maç Sayısı')

    plt.tight_layout()
    plt.show()

### 6.2 elo_ratings_wc2026.csv — Elo Derecelendirmeleri

In [ ]:
if not elo.empty:
    print(f"Toplam kayıt         : {len(elo):,}")
    print(f"Yıl aralığı          : {elo['year'].min()}  →  {elo['year'].max()}")
    print(f"Benzersiz ülke sayısı: {elo['country'].nunique()}")
    print(f"Konfederasyonlar     : {sorted(elo['confederation'].dropna().unique())}")
    print("\nEn son snapshot'taki top 20 takım:")
    latest_snap = elo['snapshot_date'].max()
    display(
        elo[elo['snapshot_date'] == latest_snap]
        .sort_values('rating', ascending=False)
        [['country', 'rank', 'rating', 'confederation']]
        .head(20)
        .reset_index(drop=True)
    )

### 6.3 GROUP_FIXTURES.CSV — 2026 Grup Fikstürü

In [ ]:
if not group_fixtures.empty:
    print(f"Toplam maç           : {len(group_fixtures)}")
    print(f"Gruplar              : {sorted(group_fixtures['group'].unique())}")
    print(f"Tarih aralığı        : {group_fixtures['date_utc'].min()}  →  {group_fixtures['date_utc'].max()}")
    print()
    for grp in sorted(group_fixtures['group'].unique()):
        sub = group_fixtures[group_fixtures['group'] == grp][['match_id','home_team','away_team','date_utc','venue']]
        print(f"  Grup {grp}: {len(sub)} maç")
    print("\nTüm fikstür:")
    display(group_fixtures)

### 6.4 KNOCKOUT_SLOTS.CSV — Eleme Turu Slotları

In [ ]:
if not knockout_slots.empty:
    print(f"Toplam slot          : {len(knockout_slots)}")
    print(f"Turlar               : {knockout_slots['round'].unique()}")
    print()
    display(knockout_slots[['match_id','round','multiplier','slot_home','slot_away','venue']])

### 6.5 goalscorers.csv — Gol Atan Oyuncular

In [ ]:
if not goalscorers.empty:
    print(f"Toplam gol kaydı     : {len(goalscorers):,}")
    print(f"Tarih aralığı        : {goalscorers['date'].min().date()}  →  {goalscorers['date'].max().date()}")
    print(f"Penaltı golü oranı   : {goalscorers['penalty'].mean()*100:.1f}%")
    print(f"Kendi golü oranı     : {goalscorers['own_goal'].mean()*100:.1f}%")
    print()
    print("Tarihsel en çok gol atan 10 oyuncu:")
    top_scorers = (
        goalscorers[goalscorers['own_goal'] == False]
        .groupby(['team', 'scorer'])
        .size()
        .reset_index(name='goals')
        .sort_values('goals', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top_scorers)

### 6.6 shootouts.csv — Penaltı Atışmaları

In [ ]:
if not shootouts.empty:
    print(f"Toplam penaltı atışması: {len(shootouts):,}")
    print(f"Tarih aralığı          : {shootouts['date'].min().date()}  →  {shootouts['date'].max().date()}")
    # İlk atışı yapan takımın kazanma oranı
    if 'first_shooter' in shootouts.columns and 'winner' in shootouts.columns:
        first_wins = (shootouts['first_shooter'] == shootouts['winner']).mean()
        print(f"İlk atan takımın kazanma oranı: {first_wins*100:.1f}%")
    display(shootouts.head())

### 6.7 former_names.csv — Eski Ülke İsimleri

In [ ]:
if not former_names.empty:
    print(f"Kayıt sayısı: {len(former_names)}")
    display(former_names)

### 6.8 players_data-2025_2026.csv — 2025/26 Oyuncu Verileri

In [ ]:
if not players.empty:
    print(f"Toplam oyuncu kaydı  : {len(players):,}")
    print(f"Kolon sayısı         : {players.shape[1]}")
    # Sadece temel kolonları göster
    base_cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Gls', 'Ast']
    available = [c for c in base_cols if c in players.columns]
    print(f"\nÖrnek kolonlar: {available}")
    print("\nİlk 5 oyuncu:")
    display(players[available].head())

    # Ligi en çok temsil eden 10 ligde oyuncu sayısı
    if 'Comp' in players.columns:
        print("\nLige göre oyuncu dağılımı (top 10):")
        display(players['Comp'].value_counts().head(10).to_frame('Oyuncu Sayısı'))

---
## 7. Genel Kalite Özeti

In [ ]:
datasets = {
    'results'       : results,
    'elo'           : elo,
    'group_fixtures': group_fixtures,
    'knockout_slots': knockout_slots,
    'goalscorers'   : goalscorers,
    'shootouts'     : shootouts,
    'former_names'  : former_names,
    'players'       : players,
}

summary_rows = []
for name, df in datasets.items():
    if df.empty:
        summary_rows.append({'dataset': name, 'rows': 0, 'cols': 0, 'missing_%': '-', 'status': '⚠️ Boş'})
        continue
    missing_pct = df.isnull().mean().mean() * 100
    status = '✅ OK' if missing_pct < 10 else ('⚠️ Yüksek eksik veri' if missing_pct < 30 else '❌ Kritik eksik veri')
    summary_rows.append({
        'dataset'   : name,
        'rows'      : f"{len(df):,}",
        'cols'      : df.shape[1],
        'missing_%' : f"{missing_pct:.1f}%",
        'status'    : status,
    })

summary_df = pd.DataFrame(summary_rows)
print("VERİ KALİTESİ ÖZETİ")
print("=" * 65)
display(summary_df.set_index('dataset'))

---
## 8. Sonraki Adım

> **`02_feature_engineering.ipynb`** → Elo birleştirme, rolling form hesaplama, saldırı/savunma gücü, upset risk skoru.

Standardize edilmiş DataFrameler (`results`, `elo`, `group_fixtures`, `knockout_slots`) bir sonraki notebook'ta doğrudan kullanılacak.